# Using Semantic Search

In [3]:
!pip install -q pandas sentence-transformers accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

class LLMBackend:
    def generate(self, prompt):
        raise NotImplementedError

class LocalLlamaColab(LLMBackend):
    """Runs Llama-3-8B locally on Colab using 4-bit quantization"""
    def __init__(self, model_id="meta-llama/Meta-Llama-3-8B-Instruct"):
        print(f"Loading {model_id} in 4-bit...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            model_kwargs={"torch_dtype": torch.float16},
        )

    def generate(self, prompt):
        messages = [
            {"role": "system", "content": "You are a folklore expert."},
            {"role": "user", "content": prompt},
        ]
        prompt_formatted = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        outputs = self.pipe(
            prompt_formatted,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
        )
        return outputs[0]["generated_text"].split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

class ThompsonMotifRAG:
    def __init__(self, csv_url):
        print("Loading Motif Database...")
        self.df = pd.read_csv(csv_url)

        self.df['search_text'] = self.df['code'].astype(str) + ": " + self.df['MOTIF'].fillna('')

        print("Encoding Motifs (this takes 1-2 mins)...")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.embedder.encode(self.df['search_text'].tolist(), show_progress_bar=True)

    def retrieve(self, query_text, top_k=15):
        query_vec = self.embedder.encode([query_text])
        similarities = cosine_similarity(query_vec, self.embeddings)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return self.df.iloc[top_indices][['code', 'MOTIF']].to_dict('records')

def extract_motifs(story_segment, rag_engine, llm_backend):
    candidates = rag_engine.retrieve(story_segment)
    candidate_str = "\n".join([f"- {c['code']}: {c['MOTIF']}" for c in candidates])

    prompt = f"""
    Analyze the story segment below and identify which of the provided Thompson Motif candidates STRICTLY appear in the text.

    STORY SEGMENT:
    "{story_segment}"

    CANDIDATE MOTIFS:
    {candidate_str}

    INSTRUCTIONS:
    1. Select ONLY motifs that are clearly present in the story.
    2. If a motif is "close" but not exact, ignore it.
    3. Output format: Code - Motif Name (Brief Explanation)
    4. If none match, say "No motifs found."
    """

    return llm_backend.generate(prompt)

In [5]:
# This creates the librarian and indexes the CSV
rag = ThompsonMotifRAG("https://github.com/KatjaMellmann/TMI_as_CSV/blob/main/tmi.csv?raw=true")

Loading Motif Database...
Encoding Motifs (this takes 1-2 mins)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1447 [00:00<?, ?it/s]

In [6]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token from your Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(hf_token)

In [7]:
# 1. Initialize the Backend (This loads the model into the GPU)
# This will take a few minutes to download the weights if it's the first time
llm = LocalLlamaColab("meta-llama/Meta-Llama-3-8B-Instruct")

# 2. Check if it worked
print("LLM is now defined and ready!")

Loading meta-llama/Meta-Llama-3-8B-Instruct in 4-bit...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


LLM is now defined and ready!


In [8]:
# ATU dataset
url = "https://raw.githubusercontent.com/j-hagedorn/trilogy/master/data/aft.csv"
df = pd.read_csv(url)

print("DataFrame 'df' has been reloaded.")
print(df.head())

DataFrame 'df' has been reloaded.
  atu_id                                         tale_title provenance  \
0   910B  The Highlander Takes Three Advices from the En...   Scotland   
1   910B                     The Prince Who Acquired Wisdom      India   
2   910B                              The Three Admonitions      Italy   
3   910B                                  The Three Advices    Ireland   
4   910B  The Three Advices Which the King with the Red ...    Ireland   

                                               notes  \
0                                                NaN   
1                                                NaN   
2                                                NaN   
3  The same story is found in The Rural Repositor...   
4                                                NaN   

                                              source  \
0  Cuthbert Bede [pseudonym for Edward Bradley], ...   
1  Cecil Henry Bompas, Folklore of the Santal Par...   
2  Thomas Freder

In [9]:
df_part = df.iloc[:250,:]

In [10]:
df_part

,atu_id,tale_title,provenance,notes,source,text,data_source,date_obtained
0,910B,The Highlander Takes Three Advices from the En...,Scotland,NaN,"Cuthbert Bede [pseudonym for Edward Bradley], ...",In one of the glens of Cantire there lived a y...,Ashliman's Folktexts,2021-03-10
1,910B,The Prince Who Acquired Wisdom,India,NaN,"Cecil Henry Bompas, Folklore of the Santal Par...",There was once a raja who had an only son and ...,Ashliman's Folktexts,2021-03-10
2,910B,The Three Admonitions,Italy,NaN,"Thomas Frederick Crane, Italian Popular Tales,...",A man once left his country to go to foreign p...,Ashliman's Folktexts,2021-03-10
3,910B,The Three Advices,Ireland,The same story is found in The Rural Repositor...,"T. Crofton Croker, 'The Three Advices: An Iris...",The stories current among the Irish peasantry ...,Ashliman's Folktexts,2021-03-10
4,910B,The Three Advices Which the King with the Red ...,Ireland,NaN,"Patrick Kennedy, Legendary Fictions of the Iri...","The name of the young chief was Illan, called ...",Ashliman's Folktexts,2021-03-10
...,...,...,...,...,...,...,...,...
245,510A,The Wonderful Birch,Russia,Lang's source: 'From the Russo-Karelian.',NaN,"The woman neither spat, nor did she run betwee...",Ashliman's Folktexts,2021-03-10
246,510B,All-Kinds-of-Fur (Grimm),"Version of 1812 Germany, Jacob and Wilhelm Grimm","Incest, one of our strongest taboos, has, unti...","Jacob and Wilhelm Grimm, 'Allerlei-Rauh,' Kind...",Once upon a time there was a king whose wife w...,Ashliman's Folktexts,2021-03-10
247,510B,All-Kinds-of-Fur (Hahn),Greece,NaN,"J. G. von Hahn, 'Allerleirauh,' Griechische un...","'How can you take me for a wife,' said the gir...",Ashliman's Folktexts,2021-03-10
248,510B,Ass'-Skin (Webster),"Basque, Wentworth Webster",Although this tale contains most motifs tradit...,"Wentworth Webster, Basque Legends, 2nd edition...","The king said to her, 'Are you like your name?...",Ashliman's Folktexts,2021-03-10


In [ ]:
df_part = df.iloc[250:500,:]

In [ ]:
df_part

,atu_id,tale_title,provenance,notes,source,text,data_source,date_obtained
250,510B,Cinder Blower (Bartsch),Karl Bartsch,NaN,"Karl Bartsch, Sagen, MÃ¤rchen und GebrÃ¤uche a...",A rich widower had an only daughter who was de...,Ashliman's Folktexts,2021-03-10
251,510B,Doralice (Straparola),"Italy, Giovanni Francesco Straparola",We know almost nothing about the personal life...,"The Facetious Nights of Straparola, vol. 1, tr...","Tebaldo, Prince of Salerno, wishes to have his...",Ashliman's Folktexts,2021-03-10
252,510B,Emperor Heinrich in Sudemer Mountain (Kuhn),Germany,Heinrich (Henry) the Fowler was born about 876...,"A. Kuhn and W. Schwartz, 'Kaiser Heinrich in S...","After his grief had subsided somewhat, he reve...",Ashliman's Folktexts,2021-03-10
253,510B,Fair Maria Wood (Crane),"Italy, Thomas Frederick Crane",NaN,"Thomas Frederick Crane, Italian Popular Tales ...",There was once a husband and wife who had but ...,Ashliman's Folktexts,2021-03-10
254,510B,Kniaz Danila Govorila (Ralston2),Russia,Ralston's source: Alexander Afanasyev.,"W. R. S. Ralston, Russian Folk-Tales (London: ...","Sometimes it is a brother, instead of a father...",Ashliman's Folktexts,2021-03-10
...,...,...,...,...,...,...,...,...
495,750A,The Woodman's Three Wishes,England,Sternberg does not give this story a title.,"Thomas Sternberg, The Dialect and Folk-lore of...",A woodman went to the forest to fell some timb...,Ashliman's Folktexts,2021-03-10
496,1288A,Johha Fails to Count the Donkey He Is Riding,Palestine,NaN,"J. E. Hanauer, Folk-Lore of the Holy Land: Mos...",When Johha grew old enough to work for his liv...,Ashliman's Folktexts,2021-03-10
497,1288A,The Hodja and His Eight Donkeys,Turkey,Link to additional tales about Nasreddin Hodja.,"Albert Wesselski, Der Hodscha Nasreddin, vol. ...",'The one you were sitting on brought the numbe...,Ashliman's Folktexts,2021-03-10
498,1288A,The Simpleton with Ten Asses,Turkey,NaN,"MÃ¢r Gregory John Bar-Hebraeus, Laughable Stor...","A simpleton, who was a servant, had ten asses ...",Ashliman's Folktexts,2021-03-10


In [ ]:
df_part = df.iloc[500:750,:]

In [ ]:
df_part

,atu_id,tale_title,provenance,notes,source,text,data_source,date_obtained
500,1287,The Five Traveling Journeymen,Germany (Swabia),Swabians are featured in many German anecdotes...,"Ernst Meier, 'Die fÃ¼nf Handwerksburschen auf ...",Five journeymen once left a particular place t...,Ashliman's Folktexts,2021-03-10
501,1287,The Lost Peasant,Kashmir,"Knowles' source: Pandit Ãnand Kol, Zaina Kada...","J. Hinton Knowles, Folk-Tales of Kashmir, 2nd ...",Ten peasants were standing on the side of the ...,Ashliman's Folktexts,2021-03-10
502,1287,The Seven Wise Men of Buneyr,Pakistan,This tale continues with addtional episodes fu...,"Charles Swynnerton, Indian Nights' Entertainme...",Seven men of Buneyr once left their native wil...,Ashliman's Folktexts,2021-03-10
503,1287,The Twelve Men of Gotham,England,"Gotham is a village in Nottinghamshire, Englan...","W. A. Clouston, The Book of Noodles: Stories o...",On a certain day there were twelve men of Goth...,Ashliman's Folktexts,2021-03-10
504,756,Tannhäuser,Germany,Aarne-Thompson-Uther type 756. Link to a copy ...,"Jacob and Wilhelm Grimm, 'Der TannhÃ¤user,' De...","Noble TannhÃ¤user, a German knight, had travel...",Ashliman's Folktexts,2021-03-10
...,...,...,...,...,...,...,...,...
745,1965,The Three Brothers,Italy,Aarne-Thompson type 1965.,"Thomas Frederick Crane, Italian Popular Tales ...",Once upon a time there were three brothers. Tw...,Ashliman's Folktexts,2021-03-10
746,285A,Of Good Advice,Gesta Romanorum,NaN,"Gesta Romanorum, translated by Charles Swan, r...","In the reign of the Emperor Fulgentius, a cert...",Ashliman's Folktexts,2021-03-10
747,285A,The Gold-Giving Snake,The Panchatantra,Links to other translations of this tale: 'The...,Pantschatantra: FÃ¼nf BÃ¼cher indischer Fabeln...,In a certain place there lived a Brahman by th...,Ashliman's Folktexts,2021-03-10
748,285A,The Man and the Serpent,Aesop,NaN,"Joseph Jacobs, The Fables of Ãsop, Selected, ...",A countryman's son by accident trod upon a ser...,Ashliman's Folktexts,2021-03-10


In [ ]:
df_part = df.iloc[750:1000,:]

In [ ]:
df_part

,atu_id,tale_title,provenance,notes,source,text,data_source,date_obtained
750,1215,An Unusual Ride,Switzerland/Germany,NaN,"Johann Peter Hebel, 'Seltsamer Spazierritt,' S...","A man was riding home on his donkey, while his...",Ashliman's Folktexts,2021-03-10
751,1215,It Is Difficult to Please Everyone,Turkey,NaN,"Ali Nouri, 'Es ist schwer, allen gerecht zu we...",After they had gone some distance they came up...,Ashliman's Folktexts,2021-03-10
752,1215,Of the Olde Man and His Sonne That Brought His...,England,Based on a book written in the fifteenth or si...,"The Hundred Merry Tales; or, Shakspeare's Jest...","Anone he mette with other, that asked hym if t...",Ashliman's Folktexts,2021-03-10
753,1215,The Lady's Nineteenth Story,Turkey,NaN,"Sheykh-Zada, The History of the Forty Vezirs; ...",In the by-gone time an old gardener had mounte...,Ashliman's Folktexts,2021-03-10
754,1215,"The Man, the Boy, and the Donkey",Aesop,NaN,"Joseph Jacobs, The Fables of Ãsop (London: Ma...","But soon they passed a group of men, one of wh...",Ashliman's Folktexts,2021-03-10
...,...,...,...,...,...,...,...,...
995,310,Parsillette,France,"This story was told by JosÃ©phine Maurel, who ...","Revue des traditions populaires, vol. 6 (1891)...",Then they were told to make a pilgrimage to ac...,Ashliman's Folktexts,2021-03-10
996,310,Parsley [Petrosinella],Italy,This story is the first tale of day two in Bas...,"Giambattista Basile, The Pentamerone; or, The ...",There was once upon a time a woman named Pasca...,Ashliman's Folktexts,2021-03-10
997,310,Persinette,France,"Charlotte-Rose de Caumont de La Force, the Fre...","Le cabinet des fÃ©es; ou, Collection choisie d...",An English translation of this tale will be po...,Ashliman's Folktexts,2021-03-10
998,310,Prunella,Italy,Lang does not identify his source of this tale...,"Andrew Lang, The Grey Fairy Book (London: Long...","Now, the orchard belonged to a witch. One day ...",Ashliman's Folktexts,2021-03-10


In [ ]:
df_part = df.iloc[1000:1250,:]

In [ ]:
df_part

,atu_id,tale_title,provenance,notes,source,text,data_source,date_obtained
1000,333,Little Red Cap,Jacob and Wilhelm Grimm,The Grimms' source for the first variant (the ...,"'RothkÃ¤ppchen,' Kinder- und HausmÃ¤rchen, 1st...","One day her mother said to her, 'Come Little R...",Ashliman's Folktexts,2021-03-10
1001,333,Little Red Hat,Italy/Austria,The Italian title of this story is 'El cappeli...,"Christian Schneller, 'Das RothhÃ¼tchen,' MÃ¤rc...",Once there was an old woman who had a granddau...,Ashliman's Folktexts,2021-03-10
1002,333,Little Red Hood,Lower Lusatia,"Lower Lusatia (German Niederlausitz, Polish Do...","A. H. Wratislaw, Sixty Folk-Tales from Exclusi...","Once upon a time, there was a little darling d...",Ashliman's Folktexts,2021-03-10
1003,333,Little Red Riding Hood,Charles Perrault,The French title of this famous tale is 'Le Pe...,"Andrew Lang, The Blue Fairy Book, 5th edition ...",Once upon a time there lived in a certain vill...,Ashliman's Folktexts,2021-03-10
1004,333,The Grandmother,France,Collected by folklorist Achille Millien (1838-...,"Conte de la mÃ¨re-grand, from a website sponso...",There was a woman who had made some bread. She...,Ashliman's Folktexts,2021-03-10
...,...,...,...,...,...,...,...,...
1245,613,The Two Brothers (Georgia),Georgia,NaN,"Marjory Wardrop, Georgian Folk Tales (London: ...",When they had gone a little way they were hung...,Ashliman's Folktexts,2021-03-10
1246,613,The Two Brothers (Schiefner),Tibet,The episode of the faithless wife's attempt to...,"F. Anton von Schiefner, Tibetan Tales: Derived...","In long past times, a king came to the throne ...",Ashliman's Folktexts,2021-03-10
1247,613,The Two Peasants,Sri Lanka,NaN,"S. Jane Goonetilleke, 'Sinhalese Folklore,' Th...",In a certain village there once lived two peas...,Ashliman's Folktexts,2021-03-10
1248,613,The Two Travelers,Germany,Link to additional stories from the Grimm brot...,"Jacob and Wilhelm Grimm, Die beiden Wanderer, ...","Mountain and valley do not meet, but the child...",Ashliman's Folktexts,2021-03-10


In [ ]:
df_part = df.iloc[1250:,:]

In [ ]:
from tqdm.auto import tqdm

def process_stories(df, text_column, rag_engine, llm_backend):
    """
    Processes all stories in a dataframe and adds a 'motifs' column.
    """
    results = []

    print(f"Starting motif extraction for {len(df)} stories...")

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        story_content = row[text_column]

        try:
            candidates = rag_engine.retrieve(story_content)

            motif_analysis = extract_motifs(story_content, rag_engine, llm_backend)
            results.append(motif_analysis)

        except Exception as e:
            print(f"Error on row {index}: {e}")
            results.append("Error during processing")

    df['extracted_motifs'] = results
    return df

process_stories(df_part, 'text', rag, llm)

df_part.to_csv("stories_with_motifs-3.csv", index=False)

display(df_part[['text', 'extracted_motifs']].head())

Starting motif extraction for 250 stories...


  0%|          | 0/250 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `e

Error on row 1051: CUDA out of memory. Tried to allocate 4.07 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.54 GiB is free. Including non-PyTorch memory, this process has 12.02 GiB memory in use. Of the allocated memory 10.16 GiB is allocated by PyTorch, and 1.72 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Error on row 1241: CUDA out of memory. Tried to allocate 2.88 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.54 GiB is free. Including non-PyTorch memory, this process has 12.02 GiB memory in use. Of the allocated memory 8.85 GiB is allocated by PyTorch, and 3.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256)

Error on row 1248: CUDA out of memory. Tried to allocate 3.48 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.54 GiB is free. Including non-PyTorch memory, this process has 12.02 GiB memory in use. Of the allocated memory 9.52 GiB is allocated by PyTorch, and 2.37 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipython-input-311410314.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['extracted_motifs'] = results


,text,extracted_motifs
1000,"One day her mother said to her, 'Come Little R...","After analyzing the story segment, I found the..."
1001,Once there was an old woman who had a granddau...,"After analyzing the story segment, I found the..."
1002,"Once upon a time, there was a little darling d...","After analyzing the story segment, I found the..."
1003,Once upon a time there lived in a certain vill...,"After analyzing the story segment, I found the..."
1004,There was a woman who had made some bread. She...,"After analyzing the story segment, I found the..."


In [ ]:
df['extracted_motifs'][0]

['J2129.3 - Getting all the eggs at once. A peasant kills his hen so that he can immediately get all the eggs she will lay during the next year.']